# Roadmap question

Ask which high-ARR Sales deals in Q3 depend on an unreleased Engineering epic.
The gateway answers with `find_related` kind `roadmap_deals`: deal, epic, and
the team that owns the epic.

`product_lead` holds the `roadmap` group, so the star rule lets that persona
read the deal and the epic. Leadership-only deals stay hidden. `employee_eng`
has no sales or roadmap grant, so the same question is empty.

Run [02-seed-sales-product-eng.ipynb](02-seed-sales-product-eng.ipynb) first
if the curated set is not loaded.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "gateway_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import gateway_demo


In [ ]:
VISIBLE = {
    "deal-northwind-q3",
    "deal-fabrikam-q3",
    "deal-adventure-q3",
    "deal-wideworld-q3",
    "deal-litware-q3",
    "deal-alpine-q3",
}
HIDDEN = {
    "deal-contoso-q3",
    "deal-tailspin-q3",
    "deal-woodgrove-q3",
    "deal-proseware-q3",
    "deal-coho-q2",
    "deal-lucerne-q4",
    "deal-graphic-q3",
}


def deal_ids(body):
    found = set()
    for path in body["paths"]:
        for step in path["steps"]:
            if step["type"] == "Deal":
                found.add(step["id"])
    return found


lead = gateway_demo.related(gateway_demo.token("product_lead"), "roadmap_deals")
seen = deal_ids(lead)
print("product_lead", sorted(seen))
for path in lead["paths"]:
    deal, epic, team = path["steps"]
    print(deal["id"], deal["account_name"], deal["arr"], "->", epic["id"], epic["status"], "->", team["id"])
if seen != VISIBLE or seen & HIDDEN:
    raise SystemExit(f"roadmap paths {sorted(seen)}")
if any(not step["account_name"] for path in lead["paths"] for step in path["steps"] if step["type"] == "Deal"):
    raise SystemExit("a deal is missing its account name")

denied = gateway_demo.related(gateway_demo.token("employee_eng"), "roadmap_deals")
print("employee_eng", denied["paths"])
if denied["paths"]:
    raise SystemExit("employee_eng should see no roadmap path")
